# RhythmMotionCV: 节奏运动跟随小游戏 (纯CV课程项目)

本项目是一个 **纯计算机视觉交互游戏**：通过摄像头实时人体关键点检测，要求玩家按节奏完成动作（举手/下蹲/左移/右移），并根据时间误差给出 `Perfect/Good/Miss` 评分。

- 技术栈: `OpenCV + MediaPipe Pose + Numpy`
- 课程相关性: 姿态估计、时序判定、实时交互、性能评估
- 不涉及AIGC生成模型，符合CV应用课程定位


## 1. 运行说明

1. 先安装依赖（如未安装）。
2. 运行核心代码单元。
3. 运行启动单元开始游戏。

按键说明：
- `Q` / `Esc`: 退出
- `R`: 游戏结束后重新开始


In [1]:
# Colab/本地依赖安装（首次运行请执行）
!pip install -q opencv-python==4.10.0.84 mediapipe==0.10.21 "numpy<2" "protobuf>=4.25.3,<5" matplotlib pandas

# 安装后请重启运行时，然后执行下行检查
# import mediapipe as mp; print(mp.__version__, hasattr(mp, 'solutions'))


^C


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.67.1 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
langchain-experimental 0.3.4 requires langchain-community<0.4.0,>=0.3.0, but you have langchain-community 0.4.1 which is incompatible.
langchain-experimental 0.3.4 requires langchain-core<0.4.0,>=0.3.28, but you have langchain-core 1.2.20 which is incompatible.
llama-index-llms-openai 0.4.7 requires openai<2,>=1.81.0, but you have openai 2.29.0 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opentelemetry-proto 1.39.1 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.


In [ ]:
"""
RhythmMotionCV: A pure computer-vision rhythm exercise game.

Run:
    python rhythm_motion_game.py

Controls:
    Q / ESC: quit
    R: restart (when game over)
"""

from __future__ import annotations

import random
import time
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Optional

import cv2
import mediapipe as mp
import numpy as np


@dataclass
class GameConfig:
    camera_id: int = 0
    frame_w: int = 960
    frame_h: int = 540
    calibrate_sec: float = 3.0
    total_play_sec: float = 60.0
    bpm: float = 86.0
    max_lives: int = 3
    good_window: float = 0.30
    perfect_window: float = 0.12
    hold_frames: int = 4
    hands_margin: float = 0.04
    squat_delta: float = 0.09
    move_delta: float = 0.07
    preview_window_sec: float = 2.6
    random_seed: int = 2026


@dataclass
class PoseSnapshot:
    visible: bool
    shoulder_y: float = 0.0
    wrist_l_y: float = 0.0
    wrist_r_y: float = 0.0
    hip_y: float = 0.0
    center_x: float = 0.0


@dataclass
class BeatEvent:
    beat_time: float
    action_key: str
    judged: bool = False
    result: str = ""
    time_error: float = 0.0


class PoseTracker:
    def __init__(self) -> None:
        self.mp_pose = mp.solutions.pose
        self.pose = self.mp_pose.Pose(
            static_image_mode=False,
            model_complexity=1,
            enable_segmentation=False,
            min_detection_confidence=0.55,
            min_tracking_confidence=0.55,
        )
        self.drawer = mp.solutions.drawing_utils

    def process(self, frame_bgr: np.ndarray, draw_skeleton: bool = True) -> PoseSnapshot:
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        result = self.pose.process(rgb)
        if not result.pose_landmarks:
            return PoseSnapshot(visible=False)

        if draw_skeleton:
            self.drawer.draw_landmarks(frame_bgr, result.pose_landmarks, self.mp_pose.POSE_CONNECTIONS)

        lm = result.pose_landmarks.landmark
        sh_l = lm[self.mp_pose.PoseLandmark.LEFT_SHOULDER.value]
        sh_r = lm[self.mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
        wr_l = lm[self.mp_pose.PoseLandmark.LEFT_WRIST.value]
        wr_r = lm[self.mp_pose.PoseLandmark.RIGHT_WRIST.value]
        hip_l = lm[self.mp_pose.PoseLandmark.LEFT_HIP.value]
        hip_r = lm[self.mp_pose.PoseLandmark.RIGHT_HIP.value]

        return PoseSnapshot(
            visible=True,
            shoulder_y=(sh_l.y + sh_r.y) * 0.5,
            wrist_l_y=wr_l.y,
            wrist_r_y=wr_r.y,
            hip_y=(hip_l.y + hip_r.y) * 0.5,
            center_x=(sh_l.x + sh_r.x + hip_l.x + hip_r.x) * 0.25,
        )

    def close(self) -> None:
        self.pose.close()


class ActionJudge:
    @staticmethod
    def is_action_active(
        snap: PoseSnapshot,
        baseline: Dict[str, float],
        action_key: str,
        cfg: GameConfig,
    ) -> bool:
        if not snap.visible:
            return False

        if action_key == "hands_up":
            return (
                snap.wrist_l_y < (snap.shoulder_y - cfg.hands_margin)
                and snap.wrist_r_y < (snap.shoulder_y - cfg.hands_margin)
            )
        if action_key == "squat":
            return snap.hip_y > (baseline["hip_y"] + cfg.squat_delta)
        if action_key == "move_left":
            return snap.center_x < (baseline["center_x"] - cfg.move_delta)
        if action_key == "move_right":
            return snap.center_x > (baseline["center_x"] + cfg.move_delta)
        return False


class RhythmMotionGame:
    ACTIONS = [
        ("hands_up", "Raise Both Hands"),
        ("squat", "Do a Squat"),
        ("move_left", "Move Left"),
        ("move_right", "Move Right"),
    ]

    ACTION_COLORS = {
        "hands_up": (70, 205, 70),
        "squat": (240, 180, 60),
        "move_left": (90, 160, 250),
        "move_right": (210, 120, 230),
    }

    def __init__(self, cfg: Optional[GameConfig] = None) -> None:
        self.cfg = cfg or GameConfig()
        random.seed(self.cfg.random_seed)

        self.pose_tracker = PoseTracker()
        self.baseline_samples: deque[tuple[float, float, float]] = deque(maxlen=160)
        self.baseline = {"center_x": 0.5, "hip_y": 0.62, "shoulder_y": 0.42}

        self.state = "calibrating"
        self.score = 0
        self.combo = 0
        self.max_combo = 0
        self.lives = self.cfg.max_lives

        self.session_start_t = 0.0
        self.play_start_t = 0.0
        self.play_end_t = 0.0

        self.beat_events: List[BeatEvent] = []
        self.current_idx = 0
        self.hold_counter = 0
        self.latest_feedback = ""
        self.feedback_until_t = 0.0

        self.metrics = {
            "perfect": 0,
            "good": 0,
            "miss": 0,
            "fps_values": [],
            "logs": [],
        }

        self.last_frame_t = time.perf_counter()
        self.fps_ema = 0.0

    def reset(self) -> None:
        self.state = "calibrating"
        self.score = 0
        self.combo = 0
        self.max_combo = 0
        self.lives = self.cfg.max_lives
        self.session_start_t = time.time()
        self.play_start_t = 0.0
        self.play_end_t = 0.0
        self.baseline_samples.clear()
        self.beat_events = []
        self.current_idx = 0
        self.hold_counter = 0
        self.latest_feedback = ""
        self.feedback_until_t = 0.0
        self.metrics = {
            "perfect": 0,
            "good": 0,
            "miss": 0,
            "fps_values": [],
            "logs": [],
        }
        self.last_frame_t = time.perf_counter()
        self.fps_ema = 0.0

    def _build_beats(self, start_t: float) -> None:
        interval = 60.0 / self.cfg.bpm
        num_beats = max(1, int(self.cfg.total_play_sec / interval))

        self.beat_events.clear()
        prev_key = ""
        for i in range(num_beats):
            options = [a for a in self.ACTIONS if a[0] != prev_key]
            action_key, _ = random.choice(options)
            prev_key = action_key
            self.beat_events.append(BeatEvent(beat_time=start_t + i * interval, action_key=action_key))

        self.current_idx = 0
        self.play_start_t = start_t
        self.play_end_t = start_t + self.cfg.total_play_sec
        self.state = "playing"

    def _judge_current_beat(self, now_t: float, snap: PoseSnapshot) -> None:
        if self.current_idx >= len(self.beat_events):
            self.state = "game_over"
            return

        event = self.beat_events[self.current_idx]
        action_active = ActionJudge.is_action_active(snap, self.baseline, event.action_key, self.cfg)
        if action_active:
            self.hold_counter += 1
        else:
            self.hold_counter = max(0, self.hold_counter - 1)

        dt = now_t - event.beat_time
        abs_dt = abs(dt)

        if self.hold_counter >= self.cfg.hold_frames and abs_dt <= self.cfg.good_window and not event.judged:
            if abs_dt <= self.cfg.perfect_window:
                result = "Perfect"
                self.score += 2
                self.metrics["perfect"] += 1
            else:
                result = "Good"
                self.score += 1
                self.metrics["good"] += 1

            event.judged = True
            event.result = result
            event.time_error = dt
            self.combo += 1
            self.max_combo = max(self.max_combo, self.combo)
            self.latest_feedback = result
            self.feedback_until_t = now_t + 0.55
            self.metrics["logs"].append(
                {"action": event.action_key, "result": result, "time_error": dt, "beat_time": event.beat_time}
            )
            self.current_idx += 1
            self.hold_counter = 0
            return

        if dt > self.cfg.good_window and not event.judged:
            event.judged = True
            event.result = "Miss"
            event.time_error = dt
            self.metrics["miss"] += 1
            self.lives -= 1
            self.combo = 0
            self.latest_feedback = "Miss"
            self.feedback_until_t = now_t + 0.55
            self.metrics["logs"].append(
                {"action": event.action_key, "result": "Miss", "time_error": dt, "beat_time": event.beat_time}
            )
            self.current_idx += 1
            self.hold_counter = 0

    def _draw_coach(self, frame: np.ndarray, action_key: str, phase: float, x0: int, y0: int) -> None:
        body = (35, 35, 35)
        accent = self.ACTION_COLORS.get(action_key, (80, 80, 80))
        sway = int(8 * np.sin(phase))

        head = (x0, y0 - 75 + sway)
        neck = (x0, y0 - 48 + sway)
        hip = (x0, y0 + sway)

        cv2.circle(frame, head, 18, accent, -1)
        cv2.line(frame, neck, hip, body, 4)

        if action_key == "hands_up":
            arm_l = (x0 - 36, y0 - 92 + sway)
            arm_r = (x0 + 36, y0 - 92 + sway)
        else:
            arm_l = (x0 - 42, y0 - 35 + sway)
            arm_r = (x0 + 42, y0 - 35 + sway)

        if action_key == "move_left":
            hip = (x0 - 18, y0 + sway)
        if action_key == "move_right":
            hip = (x0 + 18, y0 + sway)
        if action_key == "squat":
            hip = (x0, y0 + 22 + sway)

        cv2.line(frame, neck, arm_l, body, 4)
        cv2.line(frame, neck, arm_r, body, 4)
        cv2.line(frame, hip, (x0 - 30, y0 + 74 + sway), body, 4)
        cv2.line(frame, hip, (x0 + 30, y0 + 74 + sway), body, 4)

    def _draw_beat_lane(self, frame: np.ndarray, now_t: float) -> None:
        h, w = frame.shape[:2]
        cx = w // 2
        y = h - 85

        cv2.line(frame, (80, y), (w - 80, y), (220, 220, 220), 2)
        cv2.line(frame, (cx, y - 20), (cx, y + 20), (20, 20, 20), 3)

        start = self.current_idx
        end = min(len(self.beat_events), start + 10)
        for i in range(start, end):
            e = self.beat_events[i]
            dt = e.beat_time - now_t
            if abs(dt) > self.cfg.preview_window_sec:
                continue

            norm = dt / self.cfg.preview_window_sec
            px = int(cx + norm * (w * 0.4))
            col = self.ACTION_COLORS.get(e.action_key, (150, 150, 150))
            radius = 14 if i == self.current_idx else 10
            cv2.circle(frame, (px, y), radius, col, -1)
            cv2.circle(frame, (px, y), radius, (20, 20, 20), 2)

    def _draw_hud(self, frame: np.ndarray, snap: PoseSnapshot, now_t: float) -> None:
        cfg = self.cfg
        h, w = frame.shape[:2]

        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, 130), (245, 245, 245), -1)
        cv2.addWeighted(overlay, 0.82, frame, 0.18, 0, frame)

        cv2.putText(frame, f"Score: {self.score}", (18, 38), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (20, 20, 20), 2)
        cv2.putText(frame, f"Combo: {self.combo}", (18, 74), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (20, 20, 20), 2)
        cv2.putText(
            frame,
            f"Lives: {self.lives}",
            (18, 108),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.85,
            (25, 25, 210) if self.lives <= 1 else (20, 20, 20),
            2,
        )

        remain = 0.0
        if self.state == "playing":
            remain = max(0.0, self.play_end_t - now_t)
        cv2.putText(frame, f"Time: {remain:04.1f}s", (w - 220, 38), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (20, 20, 20), 2)
        cv2.putText(frame, f"FPS: {self.fps_ema:04.1f}", (w - 220, 74), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (20, 20, 20), 2)

        if self.state == "calibrating":
            left = max(0.0, cfg.calibrate_sec - (now_t - self.session_start_t))
            cv2.putText(
                frame,
                f"Calibrating... stand naturally ({left:03.1f}s)",
                (250, 46),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.85,
                (35, 35, 170),
                2,
            )
        elif self.state == "playing" and self.current_idx < len(self.beat_events):
            e = self.beat_events[self.current_idx]
            action_name = dict(self.ACTIONS)[e.action_key]
            cv2.putText(
                frame,
                f"Action: {action_name}",
                (245, 46),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.95,
                self.ACTION_COLORS.get(e.action_key, (20, 20, 20)),
                2,
            )
            next_in = e.beat_time - now_t
            cv2.putText(
                frame,
                f"Next Beat In: {next_in:+.2f}s",
                (245, 80),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.75,
                (20, 20, 20),
                2,
            )
            phase = now_t * 6.0
            self._draw_coach(frame, e.action_key, phase, w - 90, 250)
            self._draw_beat_lane(frame, now_t)

        if not snap.visible and self.state != "game_over":
            cv2.putText(
                frame,
                "Body not detected. Step back and keep upper body visible.",
                (220, 114),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.67,
                (20, 20, 220),
                2,
            )

        if now_t <= self.feedback_until_t and self.latest_feedback:
            color = (30, 200, 40)
            if self.latest_feedback == "Good":
                color = (20, 170, 220)
            elif self.latest_feedback == "Miss":
                color = (30, 30, 230)
            cv2.putText(
                frame,
                self.latest_feedback,
                (int(w * 0.42), int(h * 0.58)),
                cv2.FONT_HERSHEY_DUPLEX,
                1.5,
                color,
                3,
            )

        if self.state == "game_over":
            acc_denom = max(1, self.metrics["perfect"] + self.metrics["good"] + self.metrics["miss"])
            accuracy = (self.metrics["perfect"] + self.metrics["good"]) / acc_denom
            overlay2 = frame.copy()
            cv2.rectangle(overlay2, (140, 140), (820, 400), (236, 236, 236), -1)
            cv2.addWeighted(overlay2, 0.86, frame, 0.14, 0, frame)
            cv2.rectangle(frame, (140, 140), (820, 400), (30, 30, 30), 2)
            cv2.putText(frame, "Session Complete", (305, 205), cv2.FONT_HERSHEY_DUPLEX, 1.35, (20, 20, 20), 3)
            cv2.putText(frame, f"Score: {self.score}", (300, 255), cv2.FONT_HERSHEY_SIMPLEX, 0.95, (20, 20, 20), 2)
            cv2.putText(frame, f"Max Combo: {self.max_combo}", (300, 290), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (20, 20, 20), 2)
            cv2.putText(
                frame,
                f"Accuracy: {accuracy*100:.1f}%  (Perfect {self.metrics['perfect']} / Good {self.metrics['good']})",
                (210, 326),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.72,
                (20, 20, 20),
                2,
            )
            cv2.putText(frame, "Press R to restart | Q to quit", (275, 365), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (20, 20, 20), 2)

    def run(self) -> None:
        cap = cv2.VideoCapture(self.cfg.camera_id, cv2.CAP_DSHOW)
        if not cap.isOpened():
            cap = cv2.VideoCapture(self.cfg.camera_id)
        if not cap.isOpened():
            raise RuntimeError("Cannot open camera. Check webcam permissions.")

        cap.set(cv2.CAP_PROP_FRAME_WIDTH, self.cfg.frame_w)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, self.cfg.frame_h)

        self.reset()
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            frame = cv2.flip(frame, 1)
            frame = cv2.resize(frame, (self.cfg.frame_w, self.cfg.frame_h))

            now_perf = time.perf_counter()
            dt = max(1e-5, now_perf - self.last_frame_t)
            self.last_frame_t = now_perf
            inst_fps = 1.0 / dt
            if self.fps_ema == 0.0:
                self.fps_ema = inst_fps
            else:
                self.fps_ema = 0.92 * self.fps_ema + 0.08 * inst_fps
            self.metrics["fps_values"].append(self.fps_ema)

            now_t = time.time()
            snap = self.pose_tracker.process(frame, draw_skeleton=True)

            if self.state == "calibrating":
                if snap.visible:
                    self.baseline_samples.append((snap.center_x, snap.hip_y, snap.shoulder_y))
                if now_t - self.session_start_t >= self.cfg.calibrate_sec and len(self.baseline_samples) >= 20:
                    arr = np.array(self.baseline_samples, dtype=np.float32)
                    self.baseline["center_x"] = float(np.median(arr[:, 0]))
                    self.baseline["hip_y"] = float(np.median(arr[:, 1]))
                    self.baseline["shoulder_y"] = float(np.median(arr[:, 2]))
                    self._build_beats(start_t=now_t + 1.0)

            elif self.state == "playing":
                if now_t >= self.play_end_t or self.lives <= 0 or self.current_idx >= len(self.beat_events):
                    self.state = "game_over"
                else:
                    self._judge_current_beat(now_t, snap)

            self._draw_hud(frame, snap, now_t)
            cv2.imshow("RhythmMotionCV - Follow the Beat", frame)

            key = cv2.waitKey(1) & 0xFF
            if key in (ord("q"), 27):
                break
            if key == ord("r") and self.state == "game_over":
                self.reset()

        cap.release()
        cv2.destroyAllWindows()
        self.pose_tracker.close()

    def summary(self) -> Dict[str, float]:
        judged = self.metrics["perfect"] + self.metrics["good"] + self.metrics["miss"]
        avg_fps = float(np.mean(self.metrics["fps_values"])) if self.metrics["fps_values"] else 0.0
        return {
            "score": float(self.score),
            "max_combo": float(self.max_combo),
            "perfect": float(self.metrics["perfect"]),
            "good": float(self.metrics["good"]),
            "miss": float(self.metrics["miss"]),
            "accuracy": float((self.metrics["perfect"] + self.metrics["good"]) / max(1, judged)),
            "avg_fps": avg_fps,
        }


In [ ]:
# 启动游戏
cfg = GameConfig(
    camera_id=0,
    frame_w=960,
    frame_h=540,
    total_play_sec=60.0,
    bpm=86.0,
    max_lives=3,
)

game = RhythmMotionGame(cfg)
game.run()
summary = game.summary()
summary


In [ ]:
# 实验统计模板: 命中分布 + 动作类别命中率
import pandas as pd
import matplotlib.pyplot as plt

logs = pd.DataFrame(game.metrics['logs']) if game.metrics['logs'] else pd.DataFrame(columns=['action','result','time_error'])
if logs.empty:
    print('No logs yet. Please finish one game session first.')
else:
    display(logs.head())

    result_counts = logs['result'].value_counts().reindex(['Perfect','Good','Miss']).fillna(0)
    ax = result_counts.plot(kind='bar', color=['#4CAF50','#03A9F4','#F44336'], title='Hit Result Distribution')
    ax.set_xlabel('Result')
    ax.set_ylabel('Count')
    plt.show()

    action_acc = logs.assign(success=logs['result'].isin(['Perfect','Good']))\
                     .groupby('action')['success'].mean().sort_values(ascending=False)
    ax2 = (action_acc * 100).plot(kind='bar', color='#607D8B', title='Action Accuracy (%)')
    ax2.set_xlabel('Action')
    ax2.set_ylabel('Accuracy (%)')
    plt.ylim(0, 100)
    plt.show()

    avg_fps = float(np.mean(game.metrics['fps_values'])) if game.metrics['fps_values'] else 0.0
    print(f'Average FPS: {avg_fps:.2f}')
